# BNB MC+Overlay Truth Analysis -- ICARUSNuGraphAnalyzer

## Run command

```
lar -c nugraph_analyser_wtruth_v2.fcl -S <files_list>.list -n -1
```

where `<files_list>.list` is a subset of
`/pnfs/icarus/scratch/users/sdey2/v10_06_00_04p04/nugraph/BNB_NuGraph_v10_06_00_04p04_test/stage1/5000/files.list`
(BNB MC + cosmic overlay, WireCell/2D-detsim chain -- `stage0_run2_wcdnn_icarus_overlay_...` stage1 files).

## Output

Tree `NuGraphAnaCryoE/NuGraphHitOutput` (+ `NuGraphEventOutput`,
`NuGraphEdepOutput`, `NuGraphSliceOutput`, `NuGraphPFPOutput`).

## What was different about getting truth to work on THIS sample type

The truth-inclusive analyzer worked cleanly out of the box on the MPVMPR
pure-MC training sample (97.7% truth-match rate). Pointed at this BNB
overlay sample with no changes, it silently returned **0% truth match on
every hit** -- no crash, no error, just `truth_semantic == -1` everywhere.

Investigated and ruled out, each checked directly against this file's
actual contents rather than assumed from fcl config:
- `SimChannelModuleLabel: "merge"` -- confirmed present and correctly
  labeled in the file (`sim::SimChannels_merge__DetSim`). Not the cause.
- `ParticleInventoryService.OverrideRealData` (`true` vs `false`) -- bit
  identical result either way. Not the cause.
- `largeant` as `G4ModuleLabel` -- confirmed present and correct
  (`simb::MCParticles_largeant__G4`). Not the cause.

**Actual cause**: live `BackTrackerService::ChannelToTrackIDEs` does not
work correctly for this overlay production's timing/channel-mixing
mechanism. The production already ships a precomputed workaround: an
`art::Assns<recob::Hit, simb::MCParticle, anab::BackTrackerHitMatchingData>`
under the label `mcassociationsGausCryoE`.

**Fix**: added a second truth-matching code path to the analyzer
(`GetContributors()` + new fcl param `HitTruthAssnsLabel`). When
`HitTruthAssnsLabel: "mcassociationsGausCryoE"` is set, the analyzer reads
this precomputed Assns instead of calling live backtracking. Confirmed
working: truth-match rate went from 0% to ~11-12%, with 100% of matched
hits correctly tracing to the neutrino interaction (`truth_isnu==1`) --
exactly the expected shape for an overlay sample, where most hits are real
cosmic data with no simulation truth at all.

**Open question, not yet resolved**: why did the same general fcl approach
work for other MC+overlay samples before? Best guess: the existing
production CAFMaker chain may never have relied on live
`ChannelToTrackIDEs` for this in the first place -- it may already read
this same Assns product internally. Worth checking `sbncode/CAFMaker`
directly to confirm.

In [ ]:
import uproot
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import cmcrameri.cm as cmc
from sklearn.metrics import confusion_matrix

%matplotlib inline

# --- style: serif font, no gridlines, Managua colourscheme ---
mpl.rcParams["font.family"] = "serif"
mpl.rcParams["axes.grid"] = False
mpl.rcParams["axes.spines.top"] = False
mpl.rcParams["axes.spines.right"] = False

MANAGUA = cmc.managua
MANAGUA_R = cmc.managua_r
def managua_discrete(n, reverse=False):
    cmap = MANAGUA_R if reverse else MANAGUA
    return [cmap(x) for x in np.linspace(0.08, 0.92, n)]

score_cols = ["MIP", "HIP", "shower", "michel", "diffuse"]
class_labels = ["MIP", "HIP", "Shower", "Michel", "Diffuse"]

## Load the file

Edit `root_path` below to point at your latest BNB overlay output
(single file, 100-file batch, 500-file batch, whatever you've got).

In [ ]:
root_path = "hist_nugraph_analyser_truth_bnboverlay_assnstest.root"   # EDIT ME
tree_path = "NuGraphAnaCryoE/NuGraphHitOutput"

f = uproot.open(root_path)
hits = f[tree_path].arrays(
    ["event", "truth_semantic", "truth_isnu", "truth_g4id", "has_ng_output", *score_cols],
    library="pd",
)
events = hits  # per-event breakdown/bootstrap below group on the "event" column

n_events = hits["event"].nunique()
print(f"loaded {len(hits)} hits from {n_events} events -- {root_path}")

## Check 1 -- truth-match stats

For an overlay sample, expect the truth-match rate to be much LOWER than
a pure-MC sample (MPVMPR was 97.7%) -- most hits here are real cosmic data
with no simulation truth at all, so a low match rate is the correct,
expected result, not a regression. What matters is that among the hits
that DO match, they should track back to the neutrino interaction
(`truth_isnu==1`) at a high rate, and the two failure-mode sentinels
(`-2`, `5`) should stay small.

In [ ]:
n = len(hits)
has_output_frac = (hits["has_ng_output"] == 1).mean()
broken_chain_frac = (hits["truth_semantic"] == -2).mean()
invisible_frac = (hits["truth_semantic"] == 5).mean()
truth_matched_frac = (hits["truth_g4id"] >= 0).mean()
nu_frac_of_matched = (hits.loc[hits["truth_g4id"] >= 0, "truth_isnu"] == 1).mean() if truth_matched_frac > 0 else float("nan")

summary = pd.DataFrame({
    "check": [
        "hits with valid NG output",
        "hits with a truth match (truth_g4id >= 0) -- expect LOW for overlay",
        "of MATCHED hits, fraction from the neutrino interaction -- expect HIGH",
        "broken particle-chain hits (truth_semantic == -2) -- expect ~0",
        "invisible-particle hits (truth_semantic == 5) -- expect small",
    ],
    "fraction": [has_output_frac, truth_matched_frac, nu_frac_of_matched, broken_chain_frac, invisible_frac],
    "n_hits": [
        int((hits["has_ng_output"]==1).sum()),
        int((hits["truth_g4id"]>=0).sum()),
        int((hits["truth_isnu"]==1).sum()),
        int((hits["truth_semantic"]==-2).sum()),
        int((hits["truth_semantic"]==5).sum()),
    ],
})
summary["fraction"] = summary["fraction"].map(lambda x: f"{x:.2%}")
summary

## Check 2 -- semantic label distributions

True label (this run), NuGraph's predicted label (argmax), and Giulia's
training-truth reference (thesis Fig. 4.10, MPVMPR sample -- **not** a
like-for-like comparison for an overlay sample, since the event topology
and class balance genuinely differ; shown for shape reference only, not
as a target to match exactly).

In [ ]:
def predicted_dist(df):
    valid = df["has_ng_output"] == 1
    c = np.bincount(df.loc[valid, score_cols].values.argmax(axis=1), minlength=5)
    return c / c.sum()

plot_labels = {-1: "Noise", 0: "MIP", 1: "HIP", 2: "Shower", 3: "Michel", 4: "Diffuse"}
values = sorted(plot_labels.keys())
true_freq = np.array([(hits["truth_semantic"] == v).sum() for v in values]) / len(hits)

pred_freq = predicted_dist(hits)
pred_freq_full = np.concatenate([[np.nan], pred_freq])

# Giulia's reference (Fig. 4.10, read off by eye -- MPVMPR sample, shape reference only)
giulia_freq = np.array([0.045, 0.297, 0.190, 0.430, 0.013, 0.035])

x = np.arange(len(values))
width = 0.27
colors = managua_discrete(3)

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x - width, true_freq, width, label="True label (this run)", color=colors[0])
ax.bar(x, pred_freq_full, width, label="Predicted label (argmax, this run)", color=colors[1])
ax.bar(x + width, giulia_freq, width, label="Giulia's MPVMPR training truth (Fig. 4.10, approx., shape ref. only)", color=colors[2])

ax.set_xticks(x)
ax.set_xticklabels([plot_labels[v] for v in values])
ax.set_xlabel("semantic label")
ax.set_ylabel("frequency")
ax.set_title("Semantic label distribution: true vs. predicted vs. reference", fontsize=13)
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
plt.savefig("bnb_overlay_semantic_distribution.png", dpi=150)
plt.show()

## Check 3 -- confusion matrix

Filters to hits that are (a) valid NG predictions and (b) one of the 5
trained classes (`truth_semantic` in `[0,4]`) -- this excludes the ~88%+
of hits that are real cosmic data with no truth match at all (`-1`), plus
`-2`/`5` if present. What's left is, functionally, "hits matched to real
simulated truth" -- almost entirely the overlaid neutrino interaction.

In [ ]:
valid = (hits["has_ng_output"] == 1) & (hits["truth_semantic"].between(0, 4))
cm_hits = hits[valid].copy()
cm_hits["assigned"] = cm_hits[score_cols].values.argmax(axis=1)

print(f"hits used for confusion matrix: {len(cm_hits)} / {len(hits)} ({len(cm_hits)/len(hits):.1%})")

cm = confusion_matrix(cm_hits["truth_semantic"], cm_hits["assigned"], labels=range(5))
recall = cm / cm.sum(axis=1, keepdims=True)
precision = cm / cm.sum(axis=0, keepdims=True)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

for ax, mat, title in zip(axes, [recall, precision], ["Semantic Recall", "Semantic Precision"]):
    im = ax.imshow(mat, cmap=MANAGUA, vmin=0, vmax=1)
    for i in range(5):
        for j in range(5):
            val = mat[i, j]
            txt_color = "white" if (val > 0.6 or val < 0.15) else "black"
            ax.text(j, i, f"{val:.2f}", ha="center", va="center", color=txt_color, fontsize=11)
    ax.set_xticks(range(5)); ax.set_xticklabels(class_labels)
    ax.set_yticks(range(5)); ax.set_yticklabels(class_labels)
    ax.set_xlabel("Assigned label"); ax.set_ylabel("True label")
    ax.set_title(title, fontsize=13)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="fraction")

plt.tight_layout()
plt.savefig("bnb_overlay_confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
per_class = pd.DataFrame({
    "class": class_labels,
    "n_true": cm.sum(axis=1),
    "n_predicted": cm.sum(axis=0),
    "recall (diag)": np.diag(recall).round(3),
    "precision (diag)": np.diag(precision).round(3),
})
per_class

## Statistical rigor: event-level bootstrap

Hits within one event are correlated (same tracks/showers), not
independent samples -- resampling individual hits would understate true
uncertainty. This resamples whole EVENTS with replacement instead,
recomputing the confusion matrix per resample.

Precomputes each event's confusion-matrix contribution once, then sums
small 5x5 matrices per draw -- this matters once you're past a handful of
events (the naive "filter the full table inside the bootstrap loop"
approach is fine at 5 events, effectively hangs at hundreds/thousands).

In [ ]:
event_cms = {}
for ev, sub in events.groupby("event"):
    v = (sub["has_ng_output"] == 1) & (sub["truth_semantic"].between(0, 4))
    sub = sub[v]
    if len(sub) == 0:
        event_cms[ev] = np.zeros((5, 5), dtype=int)
        continue
    assigned = sub[score_cols].values.argmax(axis=1)
    event_cms[ev] = confusion_matrix(sub["truth_semantic"], assigned, labels=range(5))

event_ids = sorted(event_cms.keys())
rng = np.random.default_rng(0)
n_boot = 1000

boot_recall = np.full((n_boot, 5), np.nan)
boot_precision = np.full((n_boot, 5), np.nan)

for b in range(n_boot):
    sample_events = rng.choice(event_ids, size=len(event_ids), replace=True)
    cm_b = sum(event_cms[ev] for ev in sample_events)
    with np.errstate(invalid="ignore", divide="ignore"):
        boot_recall[b] = np.diag(cm_b) / cm_b.sum(axis=1)
        boot_precision[b] = np.diag(cm_b) / cm_b.sum(axis=0)

recall_lo, recall_hi = np.nanpercentile(boot_recall, [16, 84], axis=0)
prec_lo, prec_hi = np.nanpercentile(boot_precision, [16, 84], axis=0)
recall_point = np.diag(recall)
prec_point = np.diag(precision)

boot_summary = pd.DataFrame({
    "class": class_labels,
    "recall": recall_point.round(3),
    "recall 68% CI (event bootstrap)": [f"[{lo:.2f}, {hi:.2f}]" for lo, hi in zip(recall_lo, recall_hi)],
    "precision": prec_point.round(3),
    "precision 68% CI (event bootstrap)": [f"[{lo:.2f}, {hi:.2f}]" for lo, hi in zip(prec_lo, prec_hi)],
})
boot_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
colors = managua_discrete(5)

for ax, point, lo, hi, title in zip(
    axes, [recall_point, prec_point], [recall_lo, prec_lo], [recall_hi, prec_hi],
    ["Recall", "Precision"],
):
    yerr = np.vstack([point - lo, hi - point])
    yerr = np.clip(yerr, 0, None)
    ax.bar(class_labels, point, color=colors, yerr=yerr, capsize=4,
           error_kw={"elinewidth": 1.3, "ecolor": "black"})
    ax.set_ylim(0, 1)
    ax.set_ylabel(title.lower())
    ax.set_title(f"{title} with 68% event-level bootstrap CI ({len(event_ids)} events)", fontsize=12)

plt.tight_layout()
plt.savefig("bnb_overlay_bootstrap_uncertainty.png", dpi=150)
plt.show()

**Exact enumeration cross-check** (only meaningful/tractable for small
event counts, e.g. a handful of files) is intentionally not included here
-- past a dozen or so events, `C(2N-1,N)` possible resamples is
astronomically large. See `truth_analysis.ipynb` (the original MPVMPR
notebook) if you want to see that check on a small sample; for this BNB
overlay analysis at scale, the random 1000-draw bootstrap above is the
right tool.